## 1. Setup and Imports {#setup}

Let's start by importing all necessary modules and setting up the environment.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import sys
import os
warnings.filterwarnings('ignore')

import sys
import os


from AbXtract import *
from AbXtract import AntibodyDescriptorCalculator, Config, load_config
from AbXtract.sequence import (
    SequenceLiabilityAnalyzer,
    BashourDescriptorCalculator,
    PeptideDescriptorCalculator,
    AntibodyNumbering
)
from AbXtract.structure import (
    SASACalculator,
    ChargeAnalyzer,
    DSSPAnalyzer,
    PropkaAnalyzer,
    ArpeggioAnalyzer
)
from AbXtract.utils import (
    read_fasta,
    write_fasta,
    parse_sequence,
    validate_sequence,
    analysis_descriptors
)


# 2. Function to combine descriptors and class them

In [ ]:
def desc_Ab(HEAVY_SEQUENCE, LIGHT_SEQUENCE, PDB_FILE):
    
    if HEAVY_SEQUENCE:
        heavy_valid, heavy_msg = validate_sequence(HEAVY_SEQUENCE)
        heavy_numbered = numbering.number_sequence(HEAVY_SEQUENCE, 'H')  # Use VH portion only
        annotated_H, cdrs_H = numbering.get_cdr_sequences(heavy_numbered, 'H')
        heavy_profiles = numbering.get_peptide_profiles(HEAVY_SEQUENCE)
    
    if LIGHT_SEQUENCE:
        light_valid, light_msg = validate_sequence(LIGHT_SEQUENCE)
        light_numbered = numbering.number_sequence(LIGHT_SEQUENCE, 'L')  # Use VH portion only    
        annotated_L, cdrs_L = numbering.get_cdr_sequences(light_numbered, 'L')
        light_profiles = numbering.get_peptide_profiles(LIGHT_SEQUENCE)

    peptide_results = peptide_calc.calculate_all(
    heavy_sequence=HEAVY_SEQUENCE,
    light_sequence=LIGHT_SEQUENCE
    )

    sequence_results, liabilities = calc.calculate_sequence_descriptors(
    heavy_sequence=HEAVY_SEQUENCE,
    light_sequence=LIGHT_SEQUENCE,
    sequence_id="TestAb_Sequence"
    )

    structure_results_seq, structure_results_comp, df_residues, df_AA, df_Ab = calc.calculate_structure_descriptors(
    heavy_sequence=HEAVY_SEQUENCE,
    light_sequence=LIGHT_SEQUENCE,
    pdb_file=PDB_FILE,
    structure_id="TestAb_Structure"
    )
    
        
    # Ensure residue_sasa_sum column exists
    structures_results_seq = analysis_descriptors.add_residue_sasa_sum_column(structure_results_seq)

    # Get data
    liabilities_list = liabilities['liabilities'].iloc[0]
    structures_data = structures_results_seq.iloc[0]

    if HEAVY_SEQUENCE:
        df_heavy_final = analysis_descriptors.create_complete_antibody_dataframe( 0, df_residues, df_Ab, 
            HEAVY_SEQUENCE, annotated_H, heavy_profiles, 
            structures_data, liabilities_list, 'Heavy', 'imgt'
                                                           
        )
    else:
        df_light_final = None
        
    if LIGHT_SEQUENCE:
        df_light_final = analysis_descriptors.create_complete_antibody_dataframe(len(HEAVY_SEQUENCE), df_residues, df_Ab, 
            LIGHT_SEQUENCE, annotated_L, light_profiles,
            structures_data, liabilities_list, 'Light', 'imgt'
                                                            
        )
    else:
        df_light_final = None
        
    df_final = analysis_descriptors.combine_all_results(
        df_AA,
        structure_results_comp,
        sequence_results,
        peptide_results,
        heavy_valid=heavy_valid,
        light_valid=light_valid,
        cdrs_H=cdrs_H,
        cdrs_L=cdrs_L
    )

    return(df_heavy_final, df_light_final, df_final)

# Load config

In [ ]:
# default configuration
custom_config = Config()

'''
# Test custom configuration
custom_config = Config.from_dict({
    'pH': 7.4,
    'numbering_scheme': 'kabat',
    'verbose': True,
    'calculate_dssp': tool_status.get('dssp', False),
    'calculate_propka': tool_status.get('propka', False),
    'calculate_arpeggio': tool_status.get('arpeggio', False)
})
'''


# Check external tool availability
tool_status = custom_config.check_external_tools()
print("🛠️ External Tool Status:")
for tool, available in tool_status.items():
    status = "OK" if available else "Fail"
    print(f"  {tool}: {status}")


# Load classes

In [ ]:
numbering = AntibodyNumbering(scheme='imgt')
peptide_calc = PeptideDescriptorCalculator()
calc = AntibodyDescriptorCalculator(config=custom_config)

# Define path 

In [ ]:
abxtract_path = "/home/HX46_FR5/github/AbXtract"
sys.path.insert(0, abxtract_path)

# Set up test data paths
BASE_DIR = Path.cwd() 
DATA_DIR = BASE_DIR / "data" / "test"
DATA_DIR.mkdir(parents=True, exist_ok=True)


# Define test file paths
RESULTS_DIR = DATA_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)


# Input sequence and pdb

In [ ]:
# Test antibody sequences (based on therapeutic antibodies)
HEAVY_SEQUENCE_list = (
    "QVQLVQSGAEVKKPGASVKVSCKVSGGTFGRYGIHWVRQAPGKGLEWMGGISPSGGTTIYAQKFQGRVTMTEDTSTDTAYMELSSLKSEDTAVYYCAREKDGYNDDAFDIWGQGTMVTVSS",
    "QVQLVQSGAEVKKPGASVKVSCKVSGGTFGRYGIHWVRQAPGKGLEWMGGINPSGYGTIYAQKFQGRVTMTEDTSTDTAYMELSSLKSEDTAVYYCAREKDGYNDDAFDIWGQGTMVTVSS",
    "QVQLVQSGAEVKKPGASVKVSCKVSGGTFGSSAIHWVRQAPGKGLEWMGGISPSFGTAIYAQKFQGRVTMTEDTSTDTAYMELSSLKSEDTAVYYCAREKDGYNDDAFDIWGQGTMVTVSS",
    "QVQLVQSGAEVKKPGASVKVSCKVSGGTLSRYGIHWVRQAPGKGLEWMGGISPSFGTAIYAQKFQGRVTMTEDTSTDTAYMELSSLKSEDTAVYYCAREKDGYNDDAFDIWGQGTMVTVSS",
    "QVQLVQSGAEVKKPGASVKVSCKVSGGTFGRYGIHWVRQAPGKGLEWMGGISPSGGTTIYAQKFQGRVTMTEDTSTDTAYMELSSLKSEDTAVYYCAREKDGYNDDAFDIWGQGTMVTVSS",
    "QVQLVQSGAEVKKPGASVKVSCKVSGGTFSRAAIHWVRQAPGKGLEWMGGSIPMFGTTIYAQKFQGRVTMTEDTSTDTAYMELSSLKSEDTAVYYCAREKDGYNDDAFDIWGQGTMVTVSS",
    "QVQLVQSGAEVKKPGASVKVSCKVSGGTFGRYGIHWVRQAPGKGLEWMGGISPSGGTTIYAQKFQGRVTMTEDTSTDTAYMELSSLKSEDTAVYYCAREKDGYNDDAFDIWGQGTMVTVSS"
)
# Light chain: Includes realistic VL domain + human kappa constant region  
LIGHT_SEQUENCE_list = (
"DIQMTQSPSSVSASVGDRVTITCRASHSIGTYLAWYQQKPGKAPKLLIYYASRLQSGVPSRFSGSGSGTDFTLTISSLQPEDFANYYCQQADNLPFTFGGGTKVEIK",
"DIQMTQSPSSVSASVGDRVTITCRASHSIGTYLAWYQQKPGKAPKLLIYYASRLHSGVPSRFSGSGSGTDFTLTISSLQPEDFANYYCQQADNLPFTFGGGTKVEIK",
"DIQMTQSPSSVSASVGDRVTITCRASHSIGTYLAWYQQKPGKAPKLLIYYASRLQSGVPSRFSGSGSGTDFTLTISSLQPEDFANYYCQQADNLPFTFGGGTKVEIK",
"DIQMTQSPSSVSASVGDRVTITCRASHSIGTYLAWYQQKPGKAPKLLIYYASRLQSGVPSRFSGSGSGTDFTLTISSLQPEDFANYYCQQADNLPFTFGGGTKVEIK",
"DIQMTQSPSSVSASVGDRVTITCRASQDIGTYLAWYQQKPGKAPKLLIYYASRLQSGVPSRFSGSGSGTDFTLTISSLQPEDFANYYCQQADNLPFTFGGGTKVEIK",
"DIQMTQSPSSVSASVGDRVTITCRASHSIGTYLAWYQQKPGKAPKLLIYYASRLQSGVPSRFSGSGSGTDFTLTISSLQPEDFANYYCQQADNLPFTFGGGTKVEIK",
"DIQMTQSPSSVSASVGDRVTITCRASEDIGTYLAWYQQKPGKAPKLLIYYASRLQSGVPSRFSGSGSGTDFTLTISSLQPEDFANYYCQQADNLPFTFGGGTKVEIK"
)
PDB_FILE_list = [
    DATA_DIR / "AAS-41124-[41032.41124].pdb",
    DATA_DIR / "AAS-41125-[41033.41125].pdb",
    DATA_DIR / "AAS-41126-[41034.41126].pdb",
    DATA_DIR / "AAS-41127-[41035.41127].pdb",
    DATA_DIR / "AAS-41130-[41038.41130].pdb",
    DATA_DIR / "AAS-41132-[41040.41132].pdb",
    DATA_DIR / "AAS-41139-[41047.41139].pdb",
    
]

# Proper descriptors

In [ ]:
from AbXtract import AntibodyDescriptorCalculator

# Initialize calculator
calc = AntibodyDescriptorCalculator()

# Sequence validity for numbering

In [ ]:
df_fin = []
df_fin_heavy = []
df_fin_light = []

for HEAVY_SEQUENCE, LIGHT_SEQUENCE, PDB_FILE in zip(HEAVY_SEQUENCE_list, LIGHT_SEQUENCE_list, PDB_FILE_list):
    df_heavy_final, df_light_final, df_final = desc_Ab(HEAVY_SEQUENCE, LIGHT_SEQUENCE, PDB_FILE)
    df_fin.append(df_final)
    df_fin_heavy.append(df_heavy_final)
    df_fin_light.append(df_light_final)

In [ ]:
df_test = pd.concat(df_fin, axis = 0)
df_mod = analysis_descriptors.prepare_object_descriptors(df_test)
df_mod

# Clean

In [ ]:
# Remove duplicate columns (keeps the first occurrence)
df_cleaned = df_mod.loc[:, ~df_mod.columns.duplicated(keep='first')]

# Verify the cleaning worked
print(f"Original shape: {df_mod.shape}")
print(f"Cleaned shape: {df_cleaned.shape}")
print(f"Removed {df_mod.shape[1] - df_cleaned.shape[1]} duplicate columns")

# Check if Heavy_molecular_weight still exists (should be only 1 now)
heavy_cols = [col for col in df_cleaned.columns if 'Heavy_molecular_weight' in col]
print(f"Heavy_molecular_weight columns remaining: {len(heavy_cols)}")

In [ ]:
out_ = []
for col in df_mod.columns.to_list():
    if "protpy" not in col:
        out_.append(col)

In [ ]:
df_mod = df_mod[out_]

In [ ]:

ids_Ab = ["AAS-41124-[41032.41124]",
          "AAS-41125-[41033.41125]",
          "AAS-41126-[41034.41126]",
          "AAS-41127-[41035.41127]",
          "AAS-41130-[41038.41130]",
          "AAS-41132-[41040.41132]",
          "AAS-41139-[41047.41139]"]

df_mod["Identifier"] = ids_Ab

In [ ]:
df_mod

In [ ]:
df_mod.style

In [ ]:
df_mod.to_csv("./CD33_desc.csv", index = None)

# Evolution pH

In [ ]:
ID_Fv = 0


In [ ]:
df_heavy_final = df_fin_heavy[ID_Fv]
df_light_final = df_fin_light[ID_Fv]
patterns = ["Light_Charges_pH_","Heavy_Charge_pH_","Free_Energy_kcal_mol_",
            "Protein_Charge_Unfolded_","Protein_Charge_Folded_pH_",
            "pI_Folded_pH_","pI_Unfolded_pH_"]
col_ph = sorted([col for col in df_test.columns 
                 if any(pattern in col for pattern in patterns)])
object_0_df = analysis_descriptors.reshape_dataframe_by_object(df_test[col_ph])[0]

In [ ]:
object_0_df

In [ ]:
fig = analysis_descriptors.plot_ph_profiles(object_0_df, object_id=0)
plt.show()


# Residue mapping

In [ ]:
fig = analysis_descriptors.plot_protein_properties(df_heavy_final, chain_type='heavy')
plt.show()


In [ ]:
fig_heavy = analysis_descriptors.plot_protein_properties(df_light_final, chain_type='light')
plt.show()


# Propka spe

In [ ]:
fig_propka_heavy = analysis_descriptors.plot_propka_properties(df_heavy_final, chain_type='heavy')
plt.show()


In [ ]:
fig_propka_light = analysis_descriptors.plot_propka_properties(df_light_final, chain_type='light')
plt.show()

In [30]:
1


1

In [ ]:
# add a .py file to modularize the run by CLI
#have in mind we will later dockerize, not now. 
# also, it should also, parralelize the run the on cpu, so depending on the number of cpu, adapt the n jobs and depending on n jobs, distribut in N parrallel job, 
# in total do N - 2, and distribute on the N-2 core, (N-2)/(n core) per core, also add tqdm or similar to follow the overall process

- input should be csv with   
ID, sequence VH, sequence VL (if None compute VHH mode, else VH VL), path to the pdb
define --output -o and --input -i for the location of the input csv and the output folder 

also options that can be present (not mandatory) 
--mode -m as "b" "r" "mr" "mw" "wd" (each will have predefine set of parameters
                                                                
we could go even futher with those for which you should also options, 
 numbering_scheme: str = 'imgt' (you just put the 3 most popular numbering scheme)
                                                                    cdr_definition: str = 'imgt' (same as numbering scheme, should bee = numbering scheme)

    pH: float = 7.4 (like --pH -p)
    temperature: float = 25.0 (--temperatrue -t)
    is_fv: bool = True (for this one change depending if VL is None or not, so not in option but auto)
    hydrophobicity_scale: str = 'Eisenberg' (--hydrophobicity_scale or -hs)
    n_jobs: int = 1
    calculate_liabilities: bool = True
    calculate_bashour: bool = True
    calculate_peptide: bool = False # 20-30 sec
    calculate_protpy: bool = False # 7 sec 
    calculate_sasa: bool = True
    calculate_charge: bool = True
    calculate_dssp: bool = True
    calculate_propka: bool = True
    calculate_arpeggio: bool = True
    calculate_cdr_properties: bool = True
    calculate_proper: bool = True # 30-40 sec
                                                                
                                                                
                                                                
    for each of these the sets are (the default should be r)
"b" : 
    calculate_liabilities: bool = False
    calculate_bashour: bool = True
    calculate_peptide: bool = False # 20-30 sec
    calculate_protpy: bool = False # 7 sec 
    calculate_sasa: bool = False
    calculate_charge: bool = True
    calculate_dssp: bool = False
    calculate_propka: bool = False
    calculate_arpeggio: bool = False
    calculate_cdr_properties: bool = True
    calculate_proper: bool = False # 30-40 sec

                                     
"r" :
    calculate_liabilities: bool = True
    calculate_bashour: bool = True
    calculate_peptide: bool = False # 20-30 sec
    calculate_protpy: bool = False # 7 sec 
    calculate_sasa: bool = True
    calculate_charge: bool = True
    calculate_dssp: bool = True
    calculate_propka: bool = True
    calculate_arpeggio: bool = True
    calculate_cdr_properties: bool = True
    calculate_proper: bool = True # 30-40 sec

                                     
"mr" :
    calculate_liabilities: bool = True
    calculate_bashour: bool = True
    calculate_peptide: bool = False # 20-30 sec
    calculate_protpy: bool = False # 7 sec 
    calculate_sasa: bool = True
    calculate_charge: bool = True
    calculate_dssp: bool = True
    calculate_propka: bool = True
    calculate_arpeggio: bool = True
    calculate_cdr_properties: bool = True
    calculate_proper: bool = True # 30-40 sec

                                     
"mw" :
    calculate_liabilities: bool = True
    calculate_bashour: bool = True
    calculate_peptide: bool = False # 20-30 sec
    calculate_protpy: bool = True # 7 sec 
    calculate_sasa: bool = True
    calculate_charge: bool = True
    calculate_dssp: bool = True
    calculate_propka: bool = True
    calculate_arpeggio: bool = True
    calculate_cdr_properties: bool = True
    calculate_proper: bool = True # 30-40 sec

                                     
"wd" :
                                     
    calculate_liabilities: bool = True
    calculate_bashour: bool = True
    calculate_peptide: bool = True # 20-30 sec
    calculate_protpy: bool = True # 7 sec 
    calculate_sasa: bool = True
    calculate_charge: bool = True
    calculate_dssp: bool = True
    calculate_propka: bool = True
    calculate_arpeggio: bool = True
    calculate_cdr_properties: bool = True
    calculate_proper: bool = True # 30-40 sec

- you should output for any run the :
# standard full descriptors set 
df_mod
# residue specific descriptors
df_fin_heavy
df_fin_light
# pH dependent descriptors
df_test

+ and also a log file of the parameters used for computation so the input parameters defined as option, the file input name and location of output folder, so all CLI
                                     - and if any failure occured
                                     

In [ ]:
don't modify existing code, deliver .py file to make it easy to CLI

